# 0. Home Credit Default Risk — GCI Global 2026 Competition

**Objective:** predict the probability that a loan applicant will have payment difficulties (`TARGET = 1`).
**Metric:** ROC AUC. **Submission:** `SK_ID_CURR,TARGET`.

| Item | Value |
| ---- | ----- |
| Final model | _placeholder — to be filled in the final sprint (after tuning)_ |
| Current best experiment | **E10** — LightGBM (`BASE_PARAMS`), 32 raw features + 5 credit ratios − `FLAG_MOBIL` (36 features) |
| 5-fold CV AUC of E10 (mean ± std / OOF) | **0.75790 ± 0.00313 / 0.75784** (tutorial baseline E0: 0.67020 ± 0.00190) |
| Public LB AUC | _placeholder_ |

**Reproducibility:** one global `SEED = 42`, fixed `n_jobs`, frozen 5-fold split saved in `outputs/folds.npy`.
*Restart & Run All* regenerates the submission CSV bit-for-bit (verified by MD5).
This notebook is self-contained: it does not import from `src/`.

**Structure:** Market analysis (§3.0) → EDA (§3) → Machine learning (§4–§6) → Model evaluation & analysis (§7) → Business proposal (§8) → References (§9) → Experiment log (Appendix A).

> Rule compliance: only `input/train.csv` and `input/test.csv` are used (no external data);
> every test prediction comes from `predict_proba` of trained models (no hand-labeling);
> all borrowed code is cited with `# Source:` comments and in §9.

## 1.Setup

### 1.1 Import Libraries
Basic libraries, library versions and the global seed (rule R3).

In [ ]:
# Source: GCI tutorial.ipynb §1.1 — adapted: added version printout and global SEED (rule R3)
import os, random, sys, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
import seaborn as sns
import sklearn, lightgbm as lgb
warnings.filterwarnings("ignore")

SEED = 42
N_JOBS = 4  # fixed thread count for every model (multi-threading must not change results)
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
seed_everything()

print("Python", sys.version.split()[0], "|", platform.platform())
for m in [np, pd, sklearn, lgb, matplotlib, sns]:
    print(f"{m.__name__:>12}: {m.__version__}")

### 1.2 Connect with Google Drive
Works on Google Colab (mounts Drive) and locally (project root or `notebooks/`).
On Colab, change the `os.chdir` path to the folder that contains `input/`.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/gci_global/Competition")  # TODO(user): confirm path
current_dir = Path(os.getcwd())
if current_dir.name == "notebooks":          # running locally from notebooks/
    current_dir = current_dir.parent
os.chdir(current_dir)                        # all relative paths below are relative to the project root
train_file      = current_dir / "input" / "train.csv"
test_file       = current_dir / "input" / "test.csv"
sample_sub_file = current_dir / "input" / "sample_submission.csv"
OUT = current_dir / "outputs"
for d in ["oof", "test_preds", "params", "submissions"]:
    (OUT / d).mkdir(parents=True, exist_ok=True)
(current_dir / "experiments").mkdir(exist_ok=True)

Run the cell below to check if the path is correctly set.

In [ ]:
import os
from pathlib import Path

# Automatically get the current working directory
current_dir = Path(os.getcwd())

# Define file paths using pathlib
train_file = current_dir / "input" / "train.csv"
test_file = current_dir / "input" / "test.csv"
sample_sub_file = current_dir / "input" / "sample_submission.csv"

# Check if path exists
if train_file.exists() and test_file.exists() and sample_sub_file.exists():
    print("All files exist and path is correctly set.")
else:
    print("Some files are missing or path is not correctly set.")

## 2.Loading the Data

### 2.1 Data Overview
Run the cell to load the dataset as `pd.DataFrame`.

**IMPORTANT:**<br>
**When you make modifications to preprocessing or model training, always make sure to run all cells from this cell.**

In [ ]:
train = pd.read_csv(train_file)
test = pd.read_csv(test_file)
sample_sub = pd.read_csv(sample_sub_file)

# Full copies (all 32 features) for EDA / market-analysis checks; `train`/`test` are reduced in §2.2
train_full = train.copy()
test_full = test.copy()

Before conducting a full-scale analysis, we will first review a brief overview of the data.

In [ ]:
# Check train data
print(f"train shape: {train.shape}")
train.head(3)

In [ ]:
# Check test data
print(f"test shape: {test.shape}")
test.head(3)

**Data check (Sprint 1, Task 0.2):** test has the same columns as train minus `TARGET`; IDs are unique and
disjoint; `sample_submission` IDs equal the test IDs in the same order.

In [ ]:
assert list(test.columns) == [c for c in train.columns if c != "TARGET"]
assert train.SK_ID_CURR.is_unique and test.SK_ID_CURR.is_unique
assert len(set(train.SK_ID_CURR) & set(test.SK_ID_CURR)) == 0
assert (sample_sub.SK_ID_CURR.values == test.SK_ID_CURR.values).all()
print(f"All data checks passed. Target rate: {train.TARGET.mean():.5f} ({train.TARGET.sum()} positives of {len(train)})")

### 2.2 Selecting Features

It is often difficult to perform data analysis and preprocessing on all features from the beginning. Instead, an easier way to get started is to start with a small number of features and then add features one by one.

This notebook will focus on 5 features. For the remaining 25 types of features, please refer to the lecture materials, the methods introduced in this notebook, etc., and perform the analysis on your own.

Feel free to also ask questions in lectures, office hours, or in the Slack community!

In [ ]:
# Focus on 5 features
use_features = [
    "NAME_CONTRACT_TYPE",
    "AMT_INCOME_TOTAL",
    "EXT_SOURCE_2",
    "OWN_CAR_AGE",
    "ORGANIZATION_TYPE",
    # add new features to use here
]
target = train["TARGET"].values

train = train[use_features]
train["TARGET"] = target
test = test[use_features]

Let's check the data once again.

In [ ]:
# Check train data
print(f"train shape: {train.shape}")
train.head(3)

In [ ]:
# Check test data
print(f"test shape: {test.shape}")
test.head(3)

## 3.Visualizing and Understanding the Data

The first thing we need to do before building the machine learning model is to **understand the data**. We do this by visualizing and analyzing, to deepen our understanding of data distribution, missing values, outliers, correlations, and etc. The results of the analysis obtained at this stage will be useful for preprocessing, feature creation, and selection of machine learning models, which are all important to building models with better prediction ability.

### 3.0 Market Analysis

*Full text also in `reports/market_analysis.md`.*

> Scope: general domain knowledge only. No external datasets, statistics or
> articles were used. The only numbers below come from `input/train.csv`. Any
> number that is an assumption is marked *illustrative*.

#### 3.0.1. Industry overview

Consumer finance companies such as Home Credit lend to people with **little or
no formal credit history**: young adults, workers in the informal economy,
first-time borrowers and households in emerging markets. Traditional
scorecards depend heavily on a credit-bureau file. "Thin-file" and "no-file"
applicants are therefore often rejected, or offered poor terms, even though
many of them would repay reliably. A lender that can tell the reliable ones
apart from the rest using **application-time information** gains both a
commercial edge and a social one, because it widens access to credit.

Main products in this segment:

| Product | How it works | Link to this dataset |
| ------- | ------------ | -------------------- |
| **Cash loan** | Fixed amount, repaid in fixed periodic installments (annuity) | `NAME_CONTRACT_TYPE = Cash loans`, `AMT_CREDIT`, `AMT_ANNUITY` |
| **Revolving loan** | Credit line or card with a reusable limit | `NAME_CONTRACT_TYPE = Revolving loans` |
| **Point-of-sale (consumer) loan** | Financing for a specific purchase (electronics, furniture) arranged at the store | `AMT_GOODS_PRICE` (the price of the financed goods) |

#### 3.0.2. Business problem

Every approval decision can be wrong in one of two ways:

| Decision error | Business cost |
| -------------- | ------------- |
| **Approve a client who defaults** (false negative) | Loss of principal, collection costs, higher provisioning and capital requirements |
| **Reject a client who would have repaid** (false positive) | Lost interest income and customer lifetime value; a good client is excluded from credit |

The two costs are asymmetric and they change over time with funding costs,
competition and regulation. So the lender does not need a fixed yes/no
classifier. It needs a **risk ranking**: a score that orders applicants from
safest to riskiest. Management then sets an approval cut-off that matches its
current risk appetite (for example, "approve the safest 80%") and can move that
cut-off without retraining the model.

#### 3.0.3. Why ROC AUC is the right metric

- **It measures ranking, not a threshold.** AUC is the probability that a
  randomly chosen defaulter gets a higher score than a randomly chosen
  non-defaulter. That is exactly the property a risk ranking needs, because the
  cut-off is chosen later as a business decision.
- **It is robust to class imbalance.** About 8.1% of training clients have
  `TARGET = 1`. A model that predicts "no default" for everyone reaches about
  92% accuracy and is worthless. AUC gives that model 0.5.
- **It is the industry standard in disguise.** Credit-risk teams report the
  **Gini coefficient**, where Gini = 2·AUC − 1. For example, an AUC of 0.76
  corresponds to a Gini of 0.52.
- **Limitation:** AUC ignores calibration. If the scores are later used as
  probability-of-default estimates (for pricing or provisioning), they must be
  calibrated separately. This is noted for the Business Proposal.

#### 3.0.4. Stakeholders

| Stakeholder | Main interest | What they need from the model |
| ----------- | ------------- | ----------------------------- |
| Risk management | Lower default rate at a given approval rate | High, stable AUC; monitoring of drift |
| Sales / product | High approval rates, fast decisions | A score good enough to automate most decisions |
| Loan officers | Clear reasons for borderline decisions | Explanations (feature importance, SHAP) for individual cases |
| Regulators | Fair, explainable, non-discriminatory lending | Documented features; careful handling of `CODE_GENDER`, age and family status |
| Clients | Access to credit on fair terms; quick answers | Fewer wrongful rejections; transparent criteria |

#### 3.0.5. Value of a better model

1. **Better trade-off curve.** A higher AUC moves the whole approval-rate vs
   default-rate curve. At the same approval rate the lender has fewer defaults,
   or at the same default rate it can approve more clients, which means more
   revenue and more inclusion.
2. **Automation.** Applicants with clearly low or clearly high scores can be
   decided automatically. Only the uncertain middle band goes to manual review,
   which cuts cost and decision time.
3. **Risk-based terms.** Instead of a hard yes/no, risky-but-not-bad applicants
   can be offered a smaller amount, a shorter term or a larger down payment.
4. *Illustrative example (assumed numbers, not from data):* take a portfolio of
   100,000 approved loans with an average exposure of 10,000 and a 50% loss
   given default. Each percentage point of default rate removed at the same
   approval rate then saves about 100,000 × 1% × 10,000 × 50% = **5 million**
   currency units.

#### 3.0.6. Project success criteria

| Level | Criterion |
| ----- | --------- |
| Technical | 5-fold CV AUC clearly above the tutorial baseline (E0) and stable across folds (low std); leaderboard AUC consistent with CV |
| Business | The Business Proposal shows the default rate among approved clients at several approval cut-offs (e.g. 70/80/90%) for the final model vs the baseline |
| Academic | Clear hypothesis → experiment → analysis → revision trail in `experiments/experiment_log.csv`; fully reproducible notebook (identical CSV hash on re-run); all sources cited in `REFERENCES.md` |

#### 3.0.7. Data limitations and ethical considerations

- **Application-time data only.** The data has no repayment history, previous
  applications or bureau records, so the achievable AUC is capped. A realistic
  range for strong models on these 32 columns is about 0.75–0.77.
- **Black-box external scores.** `EXT_SOURCE_1/2/3` are likely to be the
  strongest predictors, but their construction is unknown, so they cannot be
  explained to clients or regulators in detail. They may also carry biases
  from their own sources.
- **Missing values carry information.** `OWN_CAR_AGE` and `EXT_SOURCE_1` are
  mostly missing, and `DAYS_EMPLOYED = 365243` is a placeholder (typically
  pensioners or unemployed). Whether a value is missing can itself be
  predictive, and it must be handled the same way in train and test.
- **Fairness.** `CODE_GENDER`, age (`DAYS_BIRTH`), `NAME_FAMILY_STATUS` and
  `CNT_CHILDREN` relate to protected or sensitive characteristics. Using them
  may raise the score, but it can produce discriminatory decisions and may be
  prohibited in some jurisdictions. The Business Proposal will report model
  performance with and without these features, and the default/approval rates
  by group.
- **Population drift.** A model trained on past applicants may degrade as the
  economy, the products or the marketing channels change. A production model
  would need monitoring (for example, a population stability index) and
  periodic retraining.

#### 3.0.8. Hypotheses from domain knowledge (tested in Sprint 2 EDA)

Each hypothesis is a testable statement about the default rate (`TARGET`
mean). The *Initial evidence* column is a one-number preview computed from
`train.csv` (Task 1.2). Quintiles Q1–Q5 run from the lowest to the highest
feature value. Overall default rate: **8.07%**.

| ID | Hypothesis (testable) | Feature(s) | Initial evidence (train) |
| -- | --------------------- | ---------- | ------------------------ |
| H-M1 | A higher payment burden (`AMT_ANNUITY / AMT_INCOME_TOTAL`) means a higher default rate | engineered ratio | annuity/income Q1 7.2% → Q5 8.4% — *supports (weak)* |
| H-M2 | A longer implied loan term (`AMT_CREDIT / AMT_ANNUITY`) means higher risk | engineered ratio | credit/annuity Q1 8.0% → Q5 6.4% — ***contradicts** at Q5 (check non-monotonicity in Sprint 2)* |
| H-M3 | Younger clients, and clients with shorter employment, have higher default rates | `DAYS_BIRTH`, `DAYS_EMPLOYED` | age (youngest→oldest) Q1 11.4% → Q5 5.2%; employment length (shortest→longest, 365243 excluded) Q1 11.0% → Q5 5.2% — *supports* |
| H-M4 | Low external scores (`EXT_SOURCE_*`) are the strongest single predictors | `EXT_SOURCE_1/2/3` | EXT_SOURCE_2 Q1 15.3% → Q5 3.5%; top univariate AUCs: EXT_SOURCE_3 0.680, EXT_SOURCE_1 0.665, EXT_SOURCE_2 0.658, DAYS_BIRTH 0.584 — *supports (univariate AUC on non-missing rows)* |
| H-M5 | Financing more than the goods price (`AMT_CREDIT > AMT_GOODS_PRICE`) signals higher risk | engineered flag | credit > goods price 8.8% (n=111021) vs credit ≤ goods price 6.8% — *supports* |
| H-M6 | Clients with recent document or phone changes (small `abs(DAYS_ID_PUBLISH)`, `abs(DAYS_LAST_PHONE_CHANGE)`) are riskier | `DAYS_ID_PUBLISH`, `DAYS_LAST_PHONE_CHANGE` | abs(DAYS_ID_PUBLISH) (most recent→oldest) Q1 10.1% → Q5 6.0%; abs(DAYS_LAST_PHONE_CHANGE) Q1 9.3% → Q5 5.7% — *supports* |
| H-M7 | Many recent credit-bureau enquiries signal credit hunger and therefore higher risk | `AMT_REQ_CREDIT_BUREAU_*` | bureau enquiries in last year: 0: 7.1%, 1-2: 7.7%, 3+: 8.2%, missing: 10.4% — *supports (weak); missingness itself is risky* |
| H-M8 | Clients in region rating 3 default more often than those in rating 1 | `REGION_RATING_CLIENT` | rating 1: 4.7%, 2: 7.9%, 3: 11.0% — *supports* |

These previews are univariate and do not establish causation. Each hypothesis
will be confirmed or rejected in the Sprint 2 EDA. The ones that hold become
candidate features for experiments E1, E2, ….

#### 3.0.9 Initial evidence for the hypotheses (Task 1.2 — preview only, full EDA in Sprint 2)

In [ ]:
# Task 1.2 - one-number previews of the market-analysis hypotheses (train only, univariate)
from sklearn.metrics import roc_auc_score

def hypothesis_evidence(df):
    d = df.copy()
    y = d["TARGET"]
    def q_rates(s, q=5):
        s = s.replace([np.inf, -np.inf], np.nan)
        m = s.notna()
        b = pd.qcut(s[m].rank(method="first"), q, labels=[f"Q{i}" for i in range(1, q + 1)])
        return y[m].groupby(b, observed=True).mean()
    def q1q5(s):
        r = q_rates(s)
        return f"Q1 {r.iloc[0]*100:.1f}% → Q5 {r.iloc[-1]*100:.1f}%"
    def uni_auc(s):
        m = s.notna()
        a = roc_auc_score(y[m], s[m])
        return max(a, 1 - a)
    emp = d["DAYS_EMPLOYED"].where(d["DAYS_EMPLOYED"] != 365243)
    age_years = -d["DAYS_BIRTH"] / 365.25
    ev = {}
    ev["H-M1"] = "annuity/income " + q1q5(d["AMT_ANNUITY"] / d["AMT_INCOME_TOTAL"])
    ev["H-M2"] = "credit/annuity " + q1q5(d["AMT_CREDIT"] / d["AMT_ANNUITY"])
    ev["H-M3"] = ("age (youngest→oldest) " + q1q5(age_years) +
                  "; employment length (shortest→longest, 365243 excluded) " + q1q5(-emp))
    num = d.drop(columns=["SK_ID_CURR", "TARGET"]).select_dtypes("number")
    aucs = {c: uni_auc(num[c]) for c in num.columns if num[c].nunique() > 1}
    top = sorted(aucs.items(), key=lambda kv: -kv[1])[:4]
    ev["H-M4"] = ("EXT_SOURCE_2 " + q1q5(d["EXT_SOURCE_2"]) + "; top univariate AUCs: " +
                  ", ".join(f"{c} {a:.3f}" for c, a in top))
    over = d["AMT_CREDIT"] > d["AMT_GOODS_PRICE"]
    known = d["AMT_GOODS_PRICE"].notna()
    ev["H-M5"] = (f"credit > goods price {y[over & known].mean()*100:.1f}% (n={int((over & known).sum())}) "
                  f"vs credit ≤ goods price {y[~over & known].mean()*100:.1f}%")
    ev["H-M6"] = ("abs(DAYS_ID_PUBLISH) (most recent→oldest) " + q1q5(d["DAYS_ID_PUBLISH"].abs()) +
                  "; abs(DAYS_LAST_PHONE_CHANGE) " + q1q5(d["DAYS_LAST_PHONE_CHANGE"].abs()))
    yr = d["AMT_REQ_CREDIT_BUREAU_YEAR"]
    grp = pd.cut(yr, [-0.5, 0.5, 2.5, np.inf], labels=["0", "1-2", "3+"])
    r = y.groupby(grp, observed=True).mean()
    ev["H-M7"] = ("bureau enquiries in last year: " + ", ".join(f"{k}: {v*100:.1f}%" for k, v in r.items()) +
                  f", missing: {y[yr.isna()].mean()*100:.1f}%")
    r = y.groupby(d["REGION_RATING_CLIENT"]).mean()
    ev["H-M8"] = "rating " + ", ".join(f"{k}: {v*100:.1f}%" for k, v in r.items())
    return ev

evidence = hypothesis_evidence(train_full)
pd.set_option("display.max_colwidth", None)
pd.DataFrame({"hypothesis": list(evidence), "initial evidence (train)": list(evidence.values())})

### 3.1 Checking missing values
In this section, we check for missing values.
This is important as **most machine learning models cannot be trained on data with missing values**. If there are missing values, they need to be filled with some value.

In [ ]:
# Check missing values of train data
train.isnull().sum()

In [ ]:
# Check missing values of test data
test.isnull().sum()

We found that there are missing values in `EXT_SOURCE_2` and `OWN_CAR_AGE`. We will deal with these missing values later. Of course, there is a possibility that there are missing values for other features that we are not covering here, so please check them by yourself.

**Findings**:<br>
* Need to deal with missing values in `EXT_SOURCE_2` and `OWN_CAR_AGE`

In [ ]:
# EDA helpers (same code as src/eda.py). Works on copies; never modifies train_full / test_full.
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
FIG_DIR = Path("reports/figures")
EDA_SEED = 42
CAT_COLS = ["NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE",
            "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE"]


def _save(fig, name):
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / name, dpi=120, bbox_inches="tight")


def wilson_ci(k, n, z=1.96):
    """95% Wilson score interval for a proportion k/n (vectorised)."""
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = k / n
    den = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return centre - half, centre + half


def rate_table(y, groups):
    """Default rate, count and Wilson CI per group (NaN group kept as 'missing')."""
    g = pd.Series(groups).astype(object).where(pd.Series(groups).notna(), "missing")
    t = pd.DataFrame({"y": np.asarray(y), "g": g.values}).groupby("g", sort=False)["y"].agg(["sum", "count"])
    t["rate"] = t["sum"] / t["count"]
    t["ci_lo"], t["ci_hi"] = wilson_ci(t["sum"], t["count"])
    return t


def a1_target_balance(train):
    rate = train["TARGET"].mean()
    fig, ax = plt.subplots(figsize=(4, 3))
    train["TARGET"].value_counts().sort_index().plot.bar(ax=ax, color=["#4C72B0", "#C44E52"])
    ax.set_title(f"A1 Target balance (default rate {rate:.2%})")
    ax.set_xlabel("TARGET"); ax.set_ylabel("rows")
    _save(fig, "A1_target_balance.png")
    return rate


def a2_missing(train):
    d = train.drop(columns=["SK_ID_CURR", "TARGET"])
    miss = d.isna().mean().sort_values(ascending=False)
    miss = miss[miss > 0]
    rows = []
    for c in miss.index:
        m = d[c].isna()
        rows.append({"column": c, "missing_%": 100 * m.mean(),
                     "default_rate_missing_%": 100 * train.loc[m, "TARGET"].mean(),
                     "default_rate_present_%": 100 * train.loc[~m, "TARGET"].mean()})
    tab = pd.DataFrame(rows).set_index("column").round(2)
    fig, ax = plt.subplots(figsize=(8, 4))
    (100 * miss).plot.bar(ax=ax, color="#4C72B0")
    ax.set_ylabel("% missing (train)"); ax.set_title("A2 Missing values per column")
    _save(fig, "A2_missing.png")
    return tab


def a3_days_employed(train):
    anom = train["DAYS_EMPLOYED"] == 365243
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].hist(train["DAYS_EMPLOYED"], bins=80, color="#4C72B0")
    axes[0].set_title("A3 DAYS_EMPLOYED (raw) — spike at 365243")
    axes[1].hist(-train.loc[~anom, "DAYS_EMPLOYED"] / 365.25, bins=60, color="#55A868")
    axes[1].set_title("Employment years (placeholder removed)")
    _save(fig, "A3_days_employed.png")
    t = pd.DataFrame({"rows": [int(anom.sum()), int((~anom).sum())],
                      "default_rate_%": [100 * train.loc[anom, "TARGET"].mean(),
                                         100 * train.loc[~anom, "TARGET"].mean()]},
                     index=["DAYS_EMPLOYED = 365243", "other"]).round(2)
    overlap = train.loc[anom, "NAME_INCOME_TYPE"].value_counts()
    return t, overlap


def a4_deciles(train):
    d = train.copy()
    emp = d["DAYS_EMPLOYED"].where(d["DAYS_EMPLOYED"] != 365243)
    feats = {
        "EXT_SOURCE_1": d["EXT_SOURCE_1"], "EXT_SOURCE_2": d["EXT_SOURCE_2"], "EXT_SOURCE_3": d["EXT_SOURCE_3"],
        "age_years": -d["DAYS_BIRTH"] / 365.25, "employment_years": -emp / 365.25,
        "annuity/income (H-M1)": d["AMT_ANNUITY"] / d["AMT_INCOME_TOTAL"],
        "credit/annuity (H-M2)": d["AMT_CREDIT"] / d["AMT_ANNUITY"],
        "credit/goods (H-M5)": d["AMT_CREDIT"] / d["AMT_GOODS_PRICE"],
    }
    fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=True)
    summary = {}
    for ax, (name, s) in zip(axes.ravel(), feats.items()):
        s = s.replace([np.inf, -np.inf], np.nan)
        m = s.notna()
        dec = pd.qcut(s[m].rank(method="first"), 10, labels=False) + 1
        t = rate_table(d.loc[m, "TARGET"], dec).sort_index()
        ax.plot(t.index, 100 * t["rate"], marker="o", color="#C44E52")
        ax.fill_between(t.index, 100 * t["ci_lo"], 100 * t["ci_hi"], alpha=0.25, color="#C44E52")
        ax.set_title(name); ax.set_xlabel("decile (low → high)"); ax.set_xticks(range(1, 11))
        summary[name] = (100 * t["rate"]).round(1).tolist()
    axes[0, 0].set_ylabel("default rate %"); axes[1, 0].set_ylabel("default rate %")
    fig.suptitle("A4 Default rate by decile (95% Wilson CI)")
    fig.tight_layout()
    _save(fig, "A4_deciles.png")
    return pd.DataFrame(summary, index=[f"D{i}" for i in range(1, 11)]).T


def a5_categoricals(train):
    fig, axes = plt.subplots(4, 2, figsize=(16, 20))
    out = {}
    for ax, c in zip(axes.ravel(), CAT_COLS):
        t = rate_table(train["TARGET"], train[c]).sort_values("count", ascending=False).head(15)
        ax2 = ax.twinx()
        ax.bar(range(len(t)), t["count"], color="#B0C4DE")
        ax2.errorbar(range(len(t)), 100 * t["rate"], yerr=[100 * (t["rate"] - t["ci_lo"]), 100 * (t["ci_hi"] - t["rate"])],
                     fmt="o", color="#C44E52")
        ax.set_xticks(range(len(t))); ax.set_xticklabels([str(i)[:22] for i in t.index], rotation=60, ha="right", fontsize=8)
        ax.set_title(c); ax.set_ylabel("count"); ax2.set_ylabel("default rate %")
        out[c] = (100 * t["rate"]).round(1)
    fig.suptitle("A5 Count (bars) and default rate with 95% CI (dots), top 15 categories")
    fig.tight_layout()
    _save(fig, "A5_categoricals.png")
    return out


def a6_correlation(train):
    num = train.drop(columns=["SK_ID_CURR", "TARGET"]).select_dtypes("number")
    num = num.loc[:, num.nunique() > 1]
    rho = num.corr(method="spearman")
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(rho, cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, xticklabels=True, yticklabels=True)
    ax.set_title("A6 Spearman correlation (numeric features)")
    _save(fig, "A6_spearman.png")
    pairs = [(a, b, rho.loc[a, b]) for i, a in enumerate(rho.columns) for b in rho.columns[i + 1:]
             if abs(rho.loc[a, b]) > 0.8]
    return pd.DataFrame(pairs, columns=["feature_a", "feature_b", "spearman_rho"]).round(3)


# Strongest raw features of the LightGBM models (gain ranking of E4/E10, restricted to raw columns)
A7_TOP_FEATURES = ["EXT_SOURCE_2", "EXT_SOURCE_3", "ORGANIZATION_TYPE", "EXT_SOURCE_1",
                   "DAYS_BIRTH", "OCCUPATION_TYPE", "DAYS_EMPLOYED", "DAYS_ID_PUBLISH"]


def raw_matrices(train, test):
    """All 32 raw features; strings -> category with levels from the sorted train+test union."""
    cols = [c for c in test.columns if c != "SK_ID_CURR"]
    both = pd.concat([train[cols], test[cols]], axis=0, ignore_index=True)
    for c in cols:
        if not pd.api.types.is_numeric_dtype(both[c]):
            both[c] = pd.Categorical(both[c], categories=sorted(both[c].dropna().unique().tolist()))
    return both.iloc[:len(train)].reset_index(drop=True), both.iloc[len(train):].reset_index(drop=True)


def a7_drift(X_tr, X_te, top_features=A7_TOP_FEATURES, n_jobs=4, seed=EDA_SEED):
    """Overlaid KDEs for the top features + adversarial validation (train=0 / test=1).

    X_tr / X_te: feature matrices with the same columns (no ID, no TARGET).
    Returns (out-of-fold AUC, top-10 features by gain share).
    """
    import lightgbm as lgb
    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    for ax, c in zip(axes.ravel(), top_features[:8]):
        if isinstance(X_tr[c].dtype, pd.CategoricalDtype):
            a = X_tr[c].value_counts(normalize=True); b = X_te[c].value_counts(normalize=True)
            top = a.index[:10]
            ax.bar(np.arange(len(top)) - 0.2, a[top], 0.4, label="train"); ax.bar(np.arange(len(top)) + 0.2, b.reindex(top), 0.4, label="test")
            ax.set_xticks(range(len(top))); ax.set_xticklabels([str(t)[:12] for t in top], rotation=60, fontsize=7)
        else:
            sns.kdeplot(X_tr[c].dropna(), ax=ax, label="train"); sns.kdeplot(X_te[c].dropna(), ax=ax, label="test")
        ax.set_title(c); ax.legend(fontsize=7)
    fig.suptitle("A7 Train vs test distributions (top 8 features of the best model)")
    fig.tight_layout()
    _save(fig, "A7_train_vs_test.png")

    X = pd.concat([X_tr, X_te], axis=0, ignore_index=True)
    z = np.r_[np.zeros(len(X_tr)), np.ones(len(X_te))].astype(int)
    oof = np.zeros(len(X)); gains = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for tr, va in skf.split(np.zeros(len(z)), z):
        m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=seed,
                               deterministic=True, force_row_wise=True, n_jobs=n_jobs, verbose=-1)
        m.fit(X.iloc[tr], z[tr])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        gains.append(pd.Series(m.booster_.feature_importance("gain"), index=m.booster_.feature_name()))
    auc = roc_auc_score(z, oof)
    top10 = pd.concat(gains, axis=1).mean(axis=1).sort_values(ascending=False).head(10)
    return auc, (100 * top10 / top10.sum()).round(1).rename("gain_share_%_of_top10")


def a8_fairness(train):
    age = pd.cut(-train["DAYS_BIRTH"] / 365.25, [0, 25, 35, 45, 55, 100], labels=["<25", "25-34", "35-44", "45-54", "55+"])
    out = {}
    for name, g in [("CODE_GENDER", train["CODE_GENDER"]), ("age band", age), ("NAME_FAMILY_STATUS", train["NAME_FAMILY_STATUS"])]:
        t = rate_table(train["TARGET"], g)
        t = t.assign(**{"share_%": 100 * t["count"] / t["count"].sum(), "default_rate_%": 100 * t["rate"],
                        "ci_95_%": [f"{100*a:.1f}–{100*b:.1f}" for a, b in zip(t["ci_lo"], t["ci_hi"])]})
        order = list(g.cat.categories) if isinstance(g.dtype, pd.CategoricalDtype) else sorted(t.index, key=str)
        out[name] = t[["count", "share_%", "default_rate_%", "ci_95_%"]].round(2).reindex([o for o in order if o in t.index])
    return out

### 3.2 Visualization and analysis of each feature (Sprint 2 EDA, A1–A8)
All figures are saved to `reports/figures/` (PNG, 120 dpi). Default rates come with 95% Wilson confidence
intervals. EDA findings are only ever turned into **features or preprocessing** applied identically to train
and test, never into rules that set predictions (R2).

> Process note: in Sprint 2 the feature experiments E1–E13 were run in the same session as this EDA, driven by
> the Sprint-1 hypotheses and their initial evidence (§3.0.9). The summary table at the end of this section
> therefore confirms after the fact which findings explain which experiment results.

#### A1 Target balance

In [ ]:
a1_rate = a1_target_balance(train_full); plt.show()
print(f'default rate: {a1_rate:.4%}')

**A1.** 8.07% of training clients defaulted (13,821 of 171,202). The data is imbalanced, so we use ROC AUC
and stratified folds. Resampling is not needed, because AUC only depends on the ranking.

#### A2 Missing values, and default rate when missing vs present

In [ ]:
a2_tab = a2_missing(train_full); plt.show()
a2_tab

**A2.** `EXT_SOURCE_1` (69.5%), `OWN_CAR_AGE` (66.0%) and `EXT_SOURCE_3` (31.9%) are mostly or often missing.
The four `AMT_REQ_CREDIT_BUREAU_*` columns are missing together (13.5%), and those clients default at
**10.4% vs 7.7%**, so missingness is informative (H-M7, input for E7). LightGBM routes NaN to its own branch,
so this signal is already usable without extra features. That is consistent with E7 being discarded
(+0.0004, below the 0.0005 threshold).

#### A3 `DAYS_EMPLOYED` placeholder 365243

In [ ]:
a3_tab, a3_overlap = a3_days_employed(train_full); plt.show()
display(a3_tab); a3_overlap

**A3.** 30,898 training rows (18.0%) have the placeholder `DAYS_EMPLOYED = 365243`. Of these, **30,888 are
pensioners and 10 unemployed**, so the value means "not employed". These clients default less (5.4% vs
8.7%). One tree split isolates the value, so replacing it with NaN plus a flag (E3) adds nothing
(Δ −0.0005, discarded).

#### A4 Default rate by decile for the key numeric features (tests H-M1 … H-M5)

In [ ]:
a4_tab = a4_deciles(train_full); plt.show()
a4_tab

**A4.**
- **H-M4 confirmed.** All three `EXT_SOURCE` scores are steep and monotonic: decile 1 is 17–20%, decile 10 is 2.4–3.1%.
- **H-M3 confirmed.** Age falls from 11.6% to 4.8%. Employment length is flat at about 11% for the shortest
  three deciles and then falls to 4.7%.
- **H-M1 is weak.** Annuity/income rises only from 6.9% to 8.9%.
- **H-M2 is rejected as a monotonic statement.** Credit/annuity (the implied term) is strongly *non-monotonic*:
  4.7%, 11.3%, …, 6.2%, **13.6%**, 5.0%. Risk depends on specific term buckets, which are probably product types.
  A linear score cannot use this, but trees can, and this is part of E4's +0.0078 gain.
- **H-M5 confirmed.** Credit/goods is flat up to decile 6 and then rises to 12.6% at decile 10 (financing well
  above the goods price).

#### A5 Categorical features: count and default rate per category

In [ ]:
a5 = a5_categoricals(train_full); plt.show()
pd.DataFrame({c: s.head(8).to_dict() for c, s in a5.items()}).T

**A5.** Categories carry large differences:
- Income type: unemployed 20%, pensioner 5.4%.
- Education: lower secondary 11.1%, higher education 5.4%.
- Housing: rented 12.9%, with parents 11.8%.
- Occupation: low-skill laborers 16.2%, accountants 4.6%.
- Organization: construction 10.9%, `XNA` 5.4% (= pensioners, see A3).

This supports keeping all 8 string columns as LightGBM `category` features (E2).

#### A6 Spearman correlation of numeric features

In [ ]:
a6_pairs = a6_correlation(train_full); plt.show()
a6_pairs

**A6.** Four pairs have |ρ| > 0.8:
- `AMT_CREDIT`–`AMT_GOODS_PRICE` 0.985
- `AMT_CREDIT`–`AMT_ANNUITY` 0.83
- `AMT_ANNUITY`–`AMT_GOODS_PRICE` 0.83
- `CNT_CHILDREN`–`CNT_FAM_MEMBERS` 0.81

The loan amounts are largely redundant with each other. What is unique is their *ratios*, which is why the
ratio features (E4) helped. Collinearity is not a problem for tree models.

#### A7 Train vs test distributions and adversarial validation

In [ ]:
Xr_tr, Xr_te = raw_matrices(train_full, test_full)
a7_auc, a7_top = a7_drift(Xr_tr, Xr_te, n_jobs=N_JOBS, seed=SEED); plt.show()
print(f'Adversarial validation AUC (train=0 / test=1): {a7_auc:.4f}')
a7_top

**A7.** The train and test distributions of the strongest features overlap almost perfectly. A LightGBM
classifier that tries to tell train rows from test rows reaches an **AUC of about 0.51**, so there is no
meaningful drift. No feature pushes it above the 0.6 warning level; `ORGANIZATION_TYPE` is the largest but
still weak contributor. CV scores should therefore transfer to the leaderboard.

#### A8 Fairness view: default rate by gender, age band and family status

In [ ]:
a8 = a8_fairness(train_full)
for k, v in a8.items():
    print(k); display(v)

**A8.**
- Gender: men default more often than women (10.1% vs 7.0%).
- Age: the rate falls from 12.1% (< 25) to 5.2% (55+).
- Family status: single and civil-marriage clients 9.9%, married 7.6%, widows 5.8%.

Sensitive attributes are therefore predictive. That makes them a fairness risk, not just a modelling choice.
Analysis experiment **E11** shows that removing `CODE_GENDER` costs **−0.0017 AUC** (0/5 folds). This trade-off
is discussed in §8.

#### EDA summary: finding → hypothesis → experiment

| Finding | Hypothesis | Experiment | Result |
| ------- | ---------- | ---------- | ------ |
| EXT_SOURCE_1/2/3 are the strongest, monotonic predictors (A4); E0 used only EXT_SOURCE_2 | H-M4 | E2 (all raw features) | **KEEP**, +0.0754 |
| Age and employment length are monotonic risk drivers (A4) | H-M3 | E2 | **KEEP** (included in E2) |
| 8 categorical columns with large rate differences (A5) | — | E2 (category dtype) | **KEEP** (included in E2) |
| `DAYS_EMPLOYED = 365243` means pensioner / not employed (A3) | H-M3 | E3 (NaN + flag) | DISCARD, −0.0005 (one split isolates it) |
| Credit/annuity strongly non-monotonic; credit/goods rises at the top (A4); amounts are collinear (A6) | H-M1, H-M2, H-M5 | E4 (credit ratios) | **KEEP**, +0.0078 |
| External scores are complementary (A4) | H-M4 | E5 (aggregates), E12 (weighted mean) | DISCARD, +0.0002 / −0.0016 |
| Life-stage ratios (age vs employment / ID / phone) | H-M3, H-M6 | E6 | DISCARD, +0.0000 |
| Missing bureau enquiries → 10.4% vs 7.7% default (A2) | H-M7 | E7 | DISCARD, +0.0004 (NaN already handled natively) |
| Family size relates to disposable income | — | E8 (per person) | DISCARD, −0.0008 |
| Peer-group position (organization / occupation / education) | — | E9 (group stats), E13 (count encoding) | DISCARD, −0.0005 / −0.0004 |
| `FLAG_MOBIL` constant, zero gain | — | E10 (drop) | **KEEP** (same AUC, simpler) |
| Gender is predictive but sensitive (A8) | fairness | E11 (drop `CODE_GENDER`) | ANALYSIS: −0.0017 AUC |
| No train/test drift (A7, AUC ≈ 0.51) | — | — | CV is trustworthy for the leaderboard |

In [ ]:
assert train_full.shape == (171202, 34) and test_full.shape == (61500, 33)  # EDA did not modify the data

## 4.Preprocessing and Feature Creation
Here, we will conduct the preprocessing and create new features based on what we have learned in the preceding visualization and analysis.

### 4.1 Baseline preprocessing (E0 — tutorial)
Baseline E0 uses **exactly the tutorial's preprocessing** of its 5 features. The only code change is
technical: with pandas ≥ 3 (Copy-on-Write), `df[col].replace(..., inplace=True)` / `.fillna(..., inplace=True)`
silently do nothing, so they are rewritten as assignments with identical meaning. The encoders are fitted
on the full train set exactly like the tutorial (no `TARGET` is used, so no leakage — approved by the user
for E0).

### NAME_CONTRACT_TYPE column
Convert `NAME_CONTRACT_TYPE` to a numeric type. In this case, “Cash loans” is converted to 0 and “Revolving loans” to 1. This method of simply replacing an integer is called **Label Encoding**.

In [ ]:
# Source: GCI tutorial.ipynb §4 — adapted: inplace replace → assignment (pandas>=3 Copy-on-Write)
# Numerization of NAME_CONTRACT_TYPE（Label Encoding）
train["NAME_CONTRACT_TYPE"] = train["NAME_CONTRACT_TYPE"].replace({'Cash loans': 0, 'Revolving loans': 1}).astype(int)
test["NAME_CONTRACT_TYPE"] = test["NAME_CONTRACT_TYPE"].replace({'Cash loans': 0, 'Revolving loans': 1}).astype(int)

train.head(5)

### ORGANIZATION_TYPE column
Convert `ORGANIZATION_TYPE` to a numeric type. This time, we will convert the variable to numeric in terms of the number of data in the variable. For example, if the number of data in “Police” is 1279 and the number of data in “Bank” is 1385, convert “Police” to 1279 and “Bank” to 1385. This method of replacing the number of data with the number of data is called **Count Encoding**.

In [ ]:
# Source: GCI tutorial.ipynb §4 — unchanged
# Numerization of ORGANIZATION_TYPE (Count Encoding）
organization_ce = train["ORGANIZATION_TYPE"].value_counts()
train["ORGANIZATION_TYPE"] = train["ORGANIZATION_TYPE"].map(organization_ce)
test["ORGANIZATION_TYPE"] = test["ORGANIZATION_TYPE"].map(organization_ce)

train.head(5)

# Sanity check: every test category was seen in train (otherwise .map would create NaN)
assert test["ORGANIZATION_TYPE"].notna().all()

### EXT_SOURCE_2 column
Fill missing values in `EXT_SOURCE_2`. There are various methods for completing missing values, but in this case, since the number of missing values is small, we simply use the average value to complete the missing values.

**IMPORANT**:
When you fill the missing values in the test data, you need to **fill with the average of the train data**.

In [ ]:
# Source: GCI tutorial.ipynb §4 — adapted: inplace fillna → assignment (pandas>=3 Copy-on-Write)
# Complete missing values of EXT_SOURCE_2 with the average
ext2_train_mean = train["EXT_SOURCE_2"].mean()
train["EXT_SOURCE_2"] = train["EXT_SOURCE_2"].fillna(ext2_train_mean)
test["EXT_SOURCE_2"] = test["EXT_SOURCE_2"].fillna(ext2_train_mean) # Use average of train data to fill test data
assert train["EXT_SOURCE_2"].notna().all() and test["EXT_SOURCE_2"].notna().all()

train.isnull().sum()

### OWN_CAR_AGE column
First, we will replace the unnatural outliers that are over 60 as `np.nan` (missing values).

In [ ]:
# Source: GCI tutorial.ipynb §4 — unchanged
# Treat values above 60 (outliers) in OWN_CAR_AGE as missing values
train.loc[train["OWN_CAR_AGE"] >= 60, "OWN_CAR_AGE"] = np.nan
test.loc[test["OWN_CAR_AGE"] >= 60, "OWN_CAR_AGE"] = np.nan

Next, we consider the handling of missing values. The original `OWN_CAR_AGE` had 112992 missing values out of 171202 data. With such a large number of missing values, it is difficult and impractical to properly fill the missing values with some value. Therefore, we will group `OWN_CAR_AGE` by decade (e.g. Group 1: 0-9 years, Group 2: 10-19 years, etc.), then apply **One Hot Encoding**.

In [ ]:
# Source: GCI tutorial.ipynb §4 — unchanged
# Divide OWN_CAR_AGE into groups
train["OWN_CAR_AGE"] = train["OWN_CAR_AGE"] // 10
test["OWN_CAR_AGE"] = test["OWN_CAR_AGE"] // 10

train["OWN_CAR_AGE"].unique()

In [ ]:
# Source: GCI tutorial.ipynb §4 — adapted: test dummies re-indexed to train's columns (safety)
# Apply One Hot Encoding to OWN_CAR_AGE
train_car_age_ohe = pd.get_dummies(train["OWN_CAR_AGE"]).add_prefix("OWN_CAR_AGE_")
test_car_age_ohe = pd.get_dummies(test["OWN_CAR_AGE"]).add_prefix("OWN_CAR_AGE_")

# Add the one hot encoded columns to train/test
train = pd.concat([train, train_car_age_ohe], axis=1)
test = pd.concat([test, test_car_age_ohe], axis=1)

# Make sure train and test have identical dummy columns in the same order
ohe_cols = list(train_car_age_ohe.columns)
assert set(test_car_age_ohe.columns) <= set(ohe_cols), "test has a car-age group unseen in train"
test = test.drop(columns=list(test_car_age_ohe.columns)).join(test_car_age_ohe.reindex(columns=ohe_cols, fill_value=False))

# Remove original OWN_CAR_AGE
train.drop('OWN_CAR_AGE', axis=1, inplace=True)
test.drop('OWN_CAR_AGE', axis=1, inplace=True)

train.head(5)

### 4.2 Feature pipeline for the LightGBM experiments (`make_features`, Sprint 2)
One function with on/off flags, used by every experiment E1–E13 (same code as `src/features.py`).
It works on the full raw data (`train_full` / `test_full`), never uses `TARGET`, and returns sorted feature
names. `FEATURE_FLAGS` below are the flags of the **current best experiment (E10)**; see Appendix A for the
experiments that were tried and discarded.

In [ ]:
# Source: own code (Sprint 2). Ratio / missing-value / EXT_SOURCE ideas are standard domain features, see §9.
FEATURE_FLAGS_DEFAULT = {
    "raw_all": False,              # E2
    "days_employed_fix": False,    # E3
    "credit_ratios": False,        # E4
    "ext_source_aggs": False,      # E5
    "time_ratios": False,          # E6
    "missing_bureau": False,       # E7
    "per_person": False,           # E8
    "group_stats": False,          # E9
    "drop_low_importance": False,  # E10 (False, or a literal list of feature names to drop)
    "drop_gender": False,          # E11 (analysis)
    "ext_source_weighted": False,  # E12 (optional)
    "count_encoding": False,       # E13 (optional)
}

ID_COL, TARGET_COL = "SK_ID_CURR", "TARGET"
EXT_COLS = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
BUREAU_COLS = ["AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_MON",
               "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_YEAR"]
DAYS_EMPLOYED_PLACEHOLDER = 365243


def _div(a, b):
    """Safe division: x / 0 -> NaN (never inf)."""
    return a / b.replace(0, np.nan)


def _tutorial_e0_features(train, test):
    """Tutorial preprocessing of its 5 features (E0/E1 base). Mirrors notebook §4 of tutorial.ipynb.

    Source: GCI tutorial.ipynb §4 — adapted: written as one function, assignment instead of inplace.
    Count encoding / mean imputation use train only, exactly like the tutorial.
    """
    use = ["NAME_CONTRACT_TYPE", "AMT_INCOME_TOTAL", "EXT_SOURCE_2", "OWN_CAR_AGE", "ORGANIZATION_TYPE"]
    tr, te = train[use].copy(), test[use].copy()
    for d in (tr, te):
        d["NAME_CONTRACT_TYPE"] = d["NAME_CONTRACT_TYPE"].replace({"Cash loans": 0, "Revolving loans": 1}).astype(int)
    ce = tr["ORGANIZATION_TYPE"].value_counts()
    tr["ORGANIZATION_TYPE"] = tr["ORGANIZATION_TYPE"].map(ce)
    te["ORGANIZATION_TYPE"] = te["ORGANIZATION_TYPE"].map(ce)
    m = tr["EXT_SOURCE_2"].mean()
    tr["EXT_SOURCE_2"] = tr["EXT_SOURCE_2"].fillna(m)
    te["EXT_SOURCE_2"] = te["EXT_SOURCE_2"].fillna(m)
    out = []
    for d in (tr, te):
        d.loc[d["OWN_CAR_AGE"] >= 60, "OWN_CAR_AGE"] = np.nan
        d["OWN_CAR_AGE"] = d["OWN_CAR_AGE"] // 10
        out.append(pd.get_dummies(d["OWN_CAR_AGE"]).add_prefix("OWN_CAR_AGE_"))
    cols = list(out[0].columns)
    assert set(out[1].columns) <= set(cols)
    tr = tr.drop(columns="OWN_CAR_AGE").join(out[0].astype(int))
    te = te.drop(columns="OWN_CAR_AGE").join(out[1].reindex(columns=cols, fill_value=False).astype(int))
    return tr, te


def make_features(train: pd.DataFrame, test: pd.DataFrame, flags: dict):
    """Returns X_train, X_test, feature_names (sorted, deterministic), categorical_feature_names.

    Applied to concat(train, test) for unsupervised steps only. Never uses TARGET.
    """
    flags = {**FEATURE_FLAGS_DEFAULT, **(flags or {})}
    unknown = set(flags) - set(FEATURE_FLAGS_DEFAULT)
    assert not unknown, f"unknown flags: {unknown}"
    n_tr = len(train)
    raw = pd.concat([train.drop(columns=[TARGET_COL], errors="ignore"), test], axis=0, ignore_index=True)
    assert (raw[ID_COL].values[:n_tr] == train[ID_COL].values).all()
    assert (raw[ID_COL].values[n_tr:] == test[ID_COL].values).all()
    raw_cols = [c for c in raw.columns if c != ID_COL]
    R = raw.copy()                       # working copy for engineered features
    feats = {}                           # name -> Series (engineered)

    # E7 / H-M7 + A2: missingness count is taken on the ORIGINAL raw values (before any fix)
    if flags["missing_bureau"]:
        feats["N_MISSING"] = raw[raw_cols].isna().sum(axis=1).astype(float)
        feats["BUREAU_REQ_MISSING"] = raw[BUREAU_COLS].isna().all(axis=1).astype(int)
        feats["BUREAU_REQ_TOTAL"] = raw[BUREAU_COLS].sum(axis=1, min_count=1)

    # E3 / H-M3 + A3: DAYS_EMPLOYED = 365243 is a placeholder (mostly pensioners) -> NaN + flag
    if flags["days_employed_fix"]:
        anom = R["DAYS_EMPLOYED"] == DAYS_EMPLOYED_PLACEHOLDER
        R["DAYS_EMPLOYED"] = R["DAYS_EMPLOYED"].where(~anom, np.nan)
        feats["DAYS_EMPLOYED_ANOM"] = anom.astype(int)

    # Base feature set
    if flags["raw_all"]:
        # E2 / H-M3, H-M4: all 32 raw features; strings -> category with levels fixed from the sorted
        # union of train+test (identical codes every run); no imputation / scaling (LightGBM handles NaN)
        base = R[raw_cols].copy()
        for c in base.columns:
            if not pd.api.types.is_numeric_dtype(base[c]):
                levels = sorted(base[c].dropna().unique().tolist())
                base[c] = pd.Categorical(base[c], categories=levels)
    else:
        # E0 / E1: tutorial's 5 features with tutorial preprocessing (10 columns)
        tr0, te0 = _tutorial_e0_features(train, test)
        base = pd.concat([tr0, te0], axis=0, ignore_index=True)

    # E4 / H-M1, H-M2, H-M5: domain ratios
    if flags["credit_ratios"]:
        feats["CREDIT_ANNUITY_RATIO"] = _div(R["AMT_CREDIT"], R["AMT_ANNUITY"])
        feats["ANNUITY_INCOME_RATIO"] = _div(R["AMT_ANNUITY"], R["AMT_INCOME_TOTAL"])
        feats["CREDIT_INCOME_RATIO"] = _div(R["AMT_CREDIT"], R["AMT_INCOME_TOTAL"])
        feats["CREDIT_GOODS_RATIO"] = _div(R["AMT_CREDIT"], R["AMT_GOODS_PRICE"])
        feats["GOODS_MINUS_CREDIT"] = R["AMT_GOODS_PRICE"] - R["AMT_CREDIT"]

    # E5 / H-M4: aggregates of the external scores (NaN-aware; product is NaN if any score is NaN)
    if flags["ext_source_aggs"]:
        E = R[EXT_COLS]
        feats["EXT_SOURCE_MEAN"] = E.mean(axis=1)
        feats["EXT_SOURCE_MIN"] = E.min(axis=1)
        feats["EXT_SOURCE_MAX"] = E.max(axis=1)
        feats["EXT_SOURCE_STD"] = E.std(axis=1, ddof=0)
        feats["EXT_SOURCE_PROD"] = E.prod(axis=1, skipna=False, min_count=3)
        feats["EXT_SOURCE_NAN_COUNT"] = E.isna().sum(axis=1).astype(float)

    # E6 / H-M3, H-M6: life-stage stability (all DAYS_* are negative, ratios are positive)
    if flags["time_ratios"]:
        age_years = -R["DAYS_BIRTH"] / 365.25
        feats["EMPLOYED_BIRTH_RATIO"] = _div(R["DAYS_EMPLOYED"], R["DAYS_BIRTH"])
        feats["ID_PUBLISH_BIRTH_RATIO"] = _div(R["DAYS_ID_PUBLISH"], R["DAYS_BIRTH"])
        feats["REGISTRATION_BIRTH_RATIO"] = _div(R["DAYS_REGISTRATION"], R["DAYS_BIRTH"])
        feats["PHONE_CHANGE_BIRTH_RATIO"] = _div(R["DAYS_LAST_PHONE_CHANGE"], R["DAYS_BIRTH"])
        feats["CAR_AGE_BIRTH_RATIO"] = _div(R["OWN_CAR_AGE"], age_years)
        # implied term in months = credit / annuity (annuity assumed monthly)
        feats["AGE_AT_LOAN_END"] = age_years + _div(R["AMT_CREDIT"], R["AMT_ANNUITY"]) / 12

    # E8: disposable income per head
    if flags["per_person"]:
        feats["INCOME_PER_PERSON"] = _div(R["AMT_INCOME_TOTAL"], R["CNT_FAM_MEMBERS"])
        feats["ANNUITY_PER_PERSON"] = _div(R["AMT_ANNUITY"], R["CNT_FAM_MEMBERS"])
        feats["CHILDREN_RATIO"] = _div(R["CNT_CHILDREN"], R["CNT_FAM_MEMBERS"])

    # E9: position relative to peer group. Group means are computed on train+test together; this is
    # an unsupervised statistic that never touches TARGET (allowed by R6 / Q6).
    if flags["group_stats"]:
        ext_f = feats["EXT_SOURCE_MEAN"] if flags["ext_source_aggs"] else R["EXT_SOURCE_2"]
        cred_f = feats["CREDIT_ANNUITY_RATIO"] if flags["credit_ratios"] else R["AMT_CREDIT"]
        cols = {"AMT_INCOME_TOTAL": R["AMT_INCOME_TOTAL"],
                ("EXT_SOURCE_MEAN" if flags["ext_source_aggs"] else "EXT_SOURCE_2"): ext_f,
                ("CREDIT_ANNUITY_RATIO" if flags["credit_ratios"] else "AMT_CREDIT"): cred_f}
        for g in ["ORGANIZATION_TYPE", "OCCUPATION_TYPE", "NAME_EDUCATION_TYPE"]:
            key = R[g].fillna("__NA__")
            for fname, f in cols.items():
                gm = f.groupby(key).transform("mean")
                feats[f"{fname}_MINUS_{g}_MEAN"] = f - gm
                feats[f"{fname}_DIV_{g}_MEAN"] = _div(f, gm)

    # E12 (optional) / H-M4: weighted external score 2*E1 + 1*E2 + 3*E3, normalised by the weights of the
    # non-missing scores (fixed formula on features; weights chosen from domain/univariate AUC, not predictions)
    if flags["ext_source_weighted"]:
        w = pd.Series({"EXT_SOURCE_1": 2.0, "EXT_SOURCE_2": 1.0, "EXT_SOURCE_3": 3.0})
        E = R[EXT_COLS]
        feats["EXT_SOURCE_WEIGHTED"] = _div((E * w).sum(axis=1, min_count=1), E.notna().mul(w).sum(axis=1))

    # E13 (optional): count encoding of the two high-cardinality categoricals, in addition to category dtype.
    # Counts are computed on train+test (unsupervised, no TARGET: allowed by R6 / Q6).
    if flags["count_encoding"]:
        for c in ["ORGANIZATION_TYPE", "OCCUPATION_TYPE"]:
            feats[f"{c}_COUNT"] = R[c].map(R[c].value_counts()).astype(float)

    X = pd.concat([base, pd.DataFrame(feats)], axis=1) if feats else base
    assert not X.columns.duplicated().any(), X.columns[X.columns.duplicated()].tolist()

    # E10: drop low-importance features (literal list, stored in the experiment's flags)
    if flags["drop_low_importance"]:
        X = X.drop(columns=[c for c in flags["drop_low_importance"] if c in X.columns])
    # E11 (analysis): exclude the sensitive attribute
    if flags["drop_gender"] and "CODE_GENDER" in X.columns:
        X = X.drop(columns="CODE_GENDER")

    assert ID_COL not in X.columns and TARGET_COL not in X.columns
    names = sorted(X.columns)
    X = X[names]
    cats = sorted(c for c in names if isinstance(X[c].dtype, pd.CategoricalDtype))
    num = X[[c for c in names if c not in cats]]
    vals = num.to_numpy(dtype=float)
    assert not np.isinf(vals).any(), "inf in numeric features"
    X_tr = X.iloc[:n_tr].reset_index(drop=True)
    X_te = X.iloc[n_tr:].reset_index(drop=True)
    assert list(X_tr.columns) == list(X_te.columns) and (X_tr.dtypes == X_te.dtypes).all()
    return X_tr, X_te, names, cats

In [ ]:
# Flags of the current best experiment E10 (literal copy of outputs/params/E10.json)
FEATURE_FLAGS = {
    **FEATURE_FLAGS_DEFAULT,
    "raw_all": True,                       # E2  KEEP
    "credit_ratios": True,                 # E4  KEEP
    "drop_low_importance": ["FLAG_MOBIL"], # E10 KEEP (zero gain / constant)
}
BEST_EXP = "E10"
X_lgb, X_test_lgb, LGB_FEATURES, LGB_CATS = make_features(train_full, test_full, FEATURE_FLAGS)
print(f"{len(LGB_FEATURES)} features, {len(LGB_CATS)} categorical: {LGB_CATS}")
X_lgb.head(3)

## 5.Building the Machine Learning Model
Now, we are ready to start building the machine learning model.

### 5.1 Import Additional Libraries
First, we import the necessary libraries for training and evaluation.

- `train_test_split`: Split data into training and evaluation data.
- `StandardScaler`: Standardize the data.
- `roc_auc_score`: Calculate ROC-AUC, the evaluation metric for this competition.
- `StratifiedKFold`: the frozen 5-fold split used for every experiment (see `docs/validation-setup.md`).

In [ ]:
# Importing libraries
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
import csv, json, datetime as dt

### 5.2 Preparing the Data
Split the data into explanatory and target variables, standardize as in the tutorial, then set up the
frozen validation framework (folds, CV functions, keep/discard rule, experiment log, artifact saving).

In [ ]:
# Source: GCI tutorial.ipynb §5.2 — adapted: keep DataFrames (column names) instead of .values
# Split the data into explanatory and target variables
X = train.drop("TARGET", axis=1)
y = train["TARGET"]
X_test = test[X.columns]
assert list(X_test.columns) == list(X.columns)
FEATURES = list(X.columns)
print(f"{len(FEATURES)} model features: {FEATURES}")

# Standardization (fitted on the full train set, exactly as in the tutorial)
sc = StandardScaler()
sc.fit(X)
X_std = pd.DataFrame(sc.transform(X), columns=FEATURES)
X_test_std = pd.DataFrame(sc.transform(X_test), columns=FEATURES)

#### Validation framework (frozen — `docs/validation-setup.md`; same code as `src/cv.py`, `src/utils.py`)

In [ ]:
# Source: docs/validation-setup.md §1-§3 (project validation spec) — adapted: load_or_make_folds and
# run_cv_sklearn added (same fold handling/outputs as run_cv_lgb)
def make_folds(y, n_splits=5, seed=SEED, path="outputs/folds.npy"):
    """Fold id (0..n_splits-1) for each training row; saved to `path` (None = don't save)."""
    folds = np.full(len(y), -1, dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for k, (_, va) in enumerate(skf.split(np.zeros(len(y)), y)):
        folds[va] = k
    assert (folds >= 0).all()
    if path is not None:
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        np.save(path, folds)
    return folds


def load_or_make_folds(y, path="outputs/folds.npy", seed=SEED):
    """Load saved folds; create them once if missing. Verifies they match a fresh regeneration."""
    path = Path(path)
    fresh = make_folds(y, seed=seed, path=None)
    if path.exists():
        folds = np.load(path)
        assert len(folds) == len(y), "folds.npy does not match the training data length"
        assert (folds == fresh).all(), "saved folds differ from make_folds(seed) - investigate"
    else:
        folds = make_folds(y, seed=seed, path=path)
    return folds


def run_cv_lgb(X, y, X_test, params, folds, seed=SEED, cat_features="auto", importance_out=None):
    """Returns oof (np.ndarray), test_pred (np.ndarray), fold_aucs (list), best_iters (list).

    Sprint 2 (approved Q3): validation data passed as eval_X/eval_y (LightGBM >= 4.6 API) instead of the
    deprecated eval_set; fold logic, early stopping and outputs are unchanged.
    importance_out: optional list; if given, one pd.Series of gain importance per fold is appended.
    """
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    fold_aucs, best_iters = [], []
    n_splits = folds.max() + 1
    for k in range(n_splits):
        tr, va = np.where(folds != k)[0], np.where(folds == k)[0]
        model = lgb.LGBMClassifier(
            **params,
            n_estimators=10000,
            random_state=seed,
            deterministic=True,
            force_row_wise=True,
            n_jobs=N_JOBS,
            verbose=-1,
        )
        model.fit(
            X.iloc[tr], y.iloc[tr],
            eval_X=(X.iloc[va],),
            eval_y=(y.iloc[va],),
            eval_metric="auc",
            categorical_feature=cat_features,
            callbacks=[lgb.early_stopping(200, verbose=False)],
        )
        oof[va] = model.predict_proba(X.iloc[va])[:, 1]
        test_pred += model.predict_proba(X_test)[:, 1] / n_splits
        fold_aucs.append(roc_auc_score(y.iloc[va], oof[va]))
        best_iters.append(model.best_iteration_)
        if importance_out is not None:
            importance_out.append(pd.Series(
                model.booster_.feature_importance(importance_type="gain"),
                index=model.booster_.feature_name()))
    print(f"Fold AUCs: {np.round(fold_aucs, 5)}")
    print(f"Mean: {np.mean(fold_aucs):.5f} ± {np.std(fold_aucs):.5f} | OOF: {roc_auc_score(y, oof):.5f}")
    return oof, test_pred, fold_aucs, best_iters


def run_cv_sklearn(model, X, y, X_test, folds):
    """Same fold handling and outputs as run_cv_lgb, for any sklearn classifier (no early stopping).

    `model` is an unfitted estimator; it is cloned for every fold (fit uses its n_jobs, predict uses
    n_jobs=1 for bit-exact reproducibility). best_iters is a list of None.
    """
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    fold_aucs, best_iters = [], []
    n_splits = folds.max() + 1
    for k in range(n_splits):
        tr, va = np.where(folds != k)[0], np.where(folds == k)[0]
        m = clone(model)
        m.fit(X.iloc[tr], y.iloc[tr])
        # Predict single-threaded: sklearn forests sum per-tree probabilities across threads in a
        # non-deterministic order, which changes the last float bit (and the CSV hash) between runs.
        if "n_jobs" in m.get_params():
            m.set_params(n_jobs=1)
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        test_pred += m.predict_proba(X_test)[:, 1] / n_splits
        fold_aucs.append(roc_auc_score(y.iloc[va], oof[va]))
        best_iters.append(None)
    print(f"Fold AUCs: {np.round(fold_aucs, 5)}")
    print(f"Mean: {np.mean(fold_aucs):.5f} ± {np.std(fold_aucs):.5f} | OOF: {roc_auc_score(y, oof):.5f}")
    return oof, test_pred, fold_aucs, best_iters


def compare(new_folds, best_folds, min_wins=4, tiny=0.0005):
    new_folds, best_folds = np.asarray(new_folds), np.asarray(best_folds)
    delta = new_folds - best_folds
    wins = int((delta > 0).sum())
    mean_delta = delta.mean()
    if mean_delta > 0 and wins >= min_wins:
        verdict = "KEEP" if mean_delta >= tiny else "CONFIRM (tiny gain → re-run with seeds 1, 2, 3)"
    else:
        verdict = "DISCARD"
    print(f"Δmean={mean_delta:+.5f} | wins={wins}/{len(delta)} | per-fold Δ={np.round(delta, 5)} → {verdict}")
    return verdict


def compare_sprint2(new_folds, best_folds, removal=False, min_wins=4, tiny=0.0005):
    """Sprint-2 override of validation-setup §3 (approved by the user for time reasons):
    - CONFIRM (0 < Δmean < tiny with >= 4/5 wins) is replaced by DISCARD.
    - removal=True (experiment only REMOVES features): KEEP if Δmean >= -0.0002 and at most 2 folds
      got worse by more than 0.0005 (simpler model, same score).
    """
    new_folds, best_folds = np.asarray(new_folds), np.asarray(best_folds)
    delta = new_folds - best_folds
    wins = int((delta > 0).sum())
    mean_delta = float(delta.mean())
    if mean_delta >= tiny and wins >= min_wins:
        verdict = "KEEP"
    elif removal and mean_delta >= -0.0002 and int((delta < -0.0005).sum()) <= 2:
        verdict = "KEEP"
    else:
        verdict = "DISCARD"
    print(f"Δmean={mean_delta:+.5f} | wins={wins}/{len(delta)} | per-fold Δ={np.round(delta, 5)} → {verdict}")
    return verdict, mean_delta, wins

In [ ]:
# Source: docs/validation-setup.md §4, §6 — adapted: log is append-only AND idempotent (no duplicate exp_id)
LOG_COLS = ["exp_id", "date", "parent_id", "hypothesis", "change", "model", "n_features",
            "fold_1", "fold_2", "fold_3", "fold_4", "fold_5", "mean_auc", "std_auc", "oof_auc",
            "delta_vs_parent", "folds_improved", "verdict", "public_lb", "notes"]


def log_experiment(row: dict, log_path="experiments/experiment_log.csv"):
    """Append one row to the experiment log (schema: validation_setup.md §4).

    Append-only and idempotent: if `exp_id` is already in the log, the row is NOT
    written again (re-running the notebook must not duplicate or overwrite history).
    Returns True if a row was written.
    """
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    new = not log_path.exists()
    if not new:
        with open(log_path, newline="") as f:
            if any(r["exp_id"] == row["exp_id"] for r in csv.DictReader(f)):
                print(f"[log] {row['exp_id']} already logged - not appended again")
                return False
    with open(log_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_COLS)
        if new:
            w.writeheader()
        w.writerow({c: row.get(c, "") for c in LOG_COLS})
    print(f"[log] {row['exp_id']} appended to {log_path}")
    return True


def save_artifacts(exp_id, oof, test_pred, params, features, best_iters, out_dir="outputs"):
    """Write OOF + test predictions (.npy) and params/features/best iterations (.json)."""
    out = Path(out_dir)
    for d in ["oof", "test_preds", "params"]:
        (out / d).mkdir(parents=True, exist_ok=True)
    np.save(out / "oof" / f"{exp_id}_oof.npy", np.asarray(oof))
    np.save(out / "test_preds" / f"{exp_id}_test.npy", np.asarray(test_pred))
    meta = {
        "exp_id": exp_id,
        "params": params,
        "features": list(features),
        "best_iters": [None if b is None else int(b) for b in best_iters],
    }
    with open(out / "params" / f"{exp_id}.json", "w") as f:
        json.dump(meta, f, indent=2, default=str)


def check_submission(path, sample_sub_path="input/sample_submission.csv"):
    """Submission sanity checks (validation_setup.md §6). Raises AssertionError on failure."""
    sub = pd.read_csv(path)
    ss = pd.read_csv(sample_sub_path)
    assert list(sub.columns) == ["SK_ID_CURR", "TARGET"]
    assert len(sub) == len(ss) and (sub.SK_ID_CURR.values == ss.SK_ID_CURR.values).all()
    assert sub.TARGET.between(0, 1).all() and sub.TARGET.notna().all()
    print(f"[check] {path}: OK ({len(sub)} rows, TARGET in [{sub.TARGET.min():.4f}, {sub.TARGET.max():.4f}])")
    return True

In [ ]:
folds = load_or_make_folds(y.values, path="outputs/folds.npy")
for k in range(5):
    print(f"fold {k}: n={(folds == k).sum()}, target rate={y[folds == k].mean():.5f}")

### 5.3 Training the Model
**Experiment E0 — baseline (harness verification).** The tutorial's final model (Random Forest,
`max_depth=10`) on the tutorial's 5 preprocessed features, evaluated with the frozen 5-fold CV instead of a
single 70/30 holdout. Only change vs the tutorial: `random_state = SEED (42)` instead of `0` (rule R3) and a
fixed `n_jobs`. The test prediction is the average of the 5 fold models' `predict_proba`.

In [ ]:
# Source: GCI tutorial.ipynb §5.3 (RandomForestClassifier(max_depth=10)) — adapted: random_state=SEED, n_jobs fixed,
# evaluated with the frozen 5-fold CV
from sklearn.ensemble import RandomForestClassifier

E0_PARAMS = dict(max_depth=10, random_state=SEED, n_jobs=N_JOBS)
rf = RandomForestClassifier(**E0_PARAMS)
e0_oof, e0_test_pred, e0_fold_aucs, e0_best_iters = run_cv_sklearn(rf, X_std, y, X_test_std, folds)

In [ ]:
log_experiment({
    "exp_id": "E0", "date": "2026-11-18", "parent_id": "-",
    "hypothesis": "Baseline: tutorial 5-feature model",
    "change": "Tutorial RF(max_depth=10) + tutorial preprocessing, evaluated in frozen 5-fold CV (random_state=42)",
    "model": "rf", "n_features": len(FEATURES),
    **{f"fold_{k+1}": round(a, 5) for k, a in enumerate(e0_fold_aucs)},
    "mean_auc": round(float(np.mean(e0_fold_aucs)), 5), "std_auc": round(float(np.std(e0_fold_aucs)), 5),
    "oof_auc": round(roc_auc_score(y, e0_oof), 5),
    "delta_vs_parent": "-", "folds_improved": "-", "verdict": "BASELINE", "public_lb": "",
    "notes": "Harness verification; RF has no early stopping so best_iters are None",
}, log_path="experiments/experiment_log.csv")
save_artifacts("E0", e0_oof, e0_test_pred, E0_PARAMS, FEATURES, e0_best_iters, out_dir="outputs")

*Debug only (not logged):* the tutorial's 70/30 holdout (with `SEED`), to sanity-check the
preprocessing was ported faithfully. A single split is never used to choose between experiments.

In [ ]:
# Source: GCI tutorial.ipynb §5.3 — adapted: random_state=SEED (rule R3); debug check only
X_train, X_valid, y_train, y_valid = train_test_split(X_std, y, test_size=0.3, stratify=y, random_state=SEED)
rf_holdout = RandomForestClassifier(random_state=SEED, max_depth=10, n_jobs=N_JOBS).fit(X_train, y_train)
print(f"Tutorial-style holdout  Train: {roc_auc_score(y_train, rf_holdout.predict_proba(X_train)[:, 1]):.5f} "
      f"| Valid: {roc_auc_score(y_valid, rf_holdout.predict_proba(X_valid)[:, 1]):.5f}")

#### Best model — E10: LightGBM on the Sprint-2 feature set
LightGBM with the fixed Sprint-2 parameters (`BASE_PARAMS`, no tuning yet), early stopping on each validation
fold (200 rounds), same frozen folds. The test prediction is the average of the 5 fold models'
`predict_proba`. The run is checked against the stored E10 artifacts from the experiment runner.

In [ ]:
# Fixed LightGBM params for Sprint 2 (docs/sprints/sprint-2.md §2) — tuning happens in Sprint 3
BASE_PARAMS = dict(
    objective="binary",
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=50,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    reg_lambda=1.0,
)
best_imps = []
best_oof, best_test_pred, best_fold_aucs, best_iters = run_cv_lgb(
    X_lgb, y, X_test_lgb, BASE_PARAMS, folds, seed=SEED, cat_features=LGB_CATS, importance_out=best_imps)
print("best iterations:", best_iters)

In [ ]:
log_experiment({
    "exp_id": BEST_EXP, "date": "2026-11-19", "parent_id": "E4",
    "hypothesis": "Fewer noisy features (zero gain / near-constant), same or better AUC",
    "change": "drop_low_importance", "model": "lgb", "n_features": len(LGB_FEATURES),
    **{f"fold_{k+1}": round(a, 5) for k, a in enumerate(best_fold_aucs)},
    "mean_auc": round(float(np.mean(best_fold_aucs)), 5), "std_auc": round(float(np.std(best_fold_aucs)), 5),
    "oof_auc": round(roc_auc_score(y, best_oof), 5), "verdict": "KEEP",
}, log_path="experiments/experiment_log.csv")   # idempotent: skipped if E10 is already logged
best_importance = pd.concat(best_imps, axis=1)
best_importance = pd.DataFrame({"gain_mean": best_importance.mean(axis=1), "gain_std": best_importance.std(axis=1)}
                               ).sort_values("gain_mean", ascending=False)
stored = OUT / "test_preds" / f"{BEST_EXP}_test.npy"
if stored.exists():   # consistency with the experiment runner (src/run_experiment.py)
    assert np.array_equal(np.load(stored), best_test_pred), "notebook result differs from the logged E10 run"
    assert np.array_equal(np.load(OUT / "oof" / f"{BEST_EXP}_oof.npy"), best_oof)
    print(f"{BEST_EXP}: notebook predictions are bit-identical to the stored experiment artifacts")
else:
    save_artifacts(BEST_EXP, best_oof, best_test_pred, {"model": "lgb", "lgb_params": BASE_PARAMS,
                   "flags": FEATURE_FLAGS}, LGB_FEATURES, best_iters, out_dir="outputs")

### 5.4 (Optional) Ensemble Learning
*Placeholder — planned for Sprint 3 (seed averaging, XGBoost, OOF-based rank-average blend).*

## 6.Creating Prediction Results
Finally, let's make a prediction for the test data, and prepare a CSV file to submit.

### 6.1 Predicting on the test data
The prediction is the average of the 5 fold models' `predict_proba` on the test data from the current best
experiment (**E10**, LightGBM). No prediction is modified after `predict_proba`. The E0 baseline prediction
(`e0_test_pred`) is kept only for comparison.

In [ ]:
# Make predictions for the test data
# Average of the 5 fold models of the current best experiment (E10)
pred = best_test_pred

### 6.2 Saving the prediction as CSV file [DO NOT CHANGE]
**WARNING**: DO **NOT** CHANGE THE CODES BELOW!!!

In [ ]:
# Put the prediction into the format of submission
sample_sub['TARGET'] = pred
sample_sub

In [ ]:
# Create the "output" directory if it doesn't exist
output_dir = current_dir / "output"
os.makedirs(output_dir, exist_ok=True)

# Specify the new output file path
output_file = output_dir / "submission.csv"

# Save the CSV file to the "output" directory
sample_sub.to_csv(output_file, index=False)

That's all for the tutorial of Home Credit Default Risk competition! Submit your CSV file to Omnicampus to see the result.

Only 5 out of 30 features are covered in this notebook, so there are a lot of room for improvement. Check out **[Next Steps]** in each section to see what you can do to improve your score.

### 6.3 Saving this notebook as a ZIP file [REQUIREMENT IF AIMING FOR HONORS/OUTSTANDING STUDENTS]
This section compresses the notebook file (`tutorial.ipynb`) into a ZIP file and saves it to your working directory.
Make sure to save the notebook (Ctrl+S or Cmd+S) before running this cell to include the latest changes.

If you are aiming to become **Honors or Outstanding Student**, you need to submit the ZIP file as well on Omnicampus.

In [ ]:
import zipfile

# Define the file names
notebook_filename = "main.ipynb" # Enter the name of this notebook
zip_filename = "main.zip"

# Define the full paths using current_dir
notebook_path = current_dir / notebook_filename
zip_output_path = current_dir / zip_filename

# Check if the notebook file exists and create the ZIP file
if notebook_path.exists():
    with zipfile.ZipFile(zip_output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add the notebook to the ZIP file (arcname avoids including the full folder structure)
        zipf.write(notebook_path, arcname=notebook_filename)
    print(f"Successfully created: {zip_output_path}")
else:
    print(f"Error: Could not find '{notebook_filename}' in {current_dir}.")
    print("Please check if the notebook name and the current directory are correct.")

### 6.4 Archive the submission and run sanity checks (added; does not change 6.2)
Copies `output/submission.csv` (written by §6.2) to `outputs/submissions/<BEST_EXP>_submission.csv` and runs the
submission sanity checks from `docs/validation-setup.md` §6.

In [ ]:
import shutil, hashlib
CURRENT_EXP = BEST_EXP
archived = OUT / "submissions" / f"{CURRENT_EXP}_submission.csv"
shutil.copyfile(output_file, archived)
check_submission(archived, sample_sub_path=sample_sub_file)
print("MD5:", hashlib.md5(archived.read_bytes()).hexdigest())

## 7. Model Evaluation & Analysis
*Full evaluation (SHAP via LightGBM `pred_contrib=True`, error analysis, approval-rate curves) in Sprint 4.*

#### 7.1 Preview: top 20 features of E10 by mean gain over the 5 folds

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
top20 = best_importance.head(20)[::-1]
ax.barh(top20.index, top20["gain_mean"], xerr=top20["gain_std"], color="#4C72B0")
ax.set_xlabel("mean gain (± std over folds)"); ax.set_title(f"{BEST_EXP}: top 20 features by gain")
fig.savefig("reports/figures/E10_top20_gain.png", dpi=120, bbox_inches="tight"); plt.show()
best_importance.head(20).round(1)

## 8. Business Proposal
*Placeholder — approval-rate vs default-rate table at several cut-offs (final model vs E0 baseline),
automation bands, risk-based terms, fairness discussion (later sprints).*

## 9. References

Every external source of code or ideas used in this project. Code adapted from a
source also carries a `# Source:` comment directly above it (rule R4).

| # | Title | Author | URL | What was used | Where in notebook |
| - | ----- | ------ | --- | ------------- | ----------------- |
| 1 | GCI Global 2026 In-class Competition Tutorial (`tutorial.ipynb`, local copy in `docs/tutorial.ipynb`) | GCI organizers | Distributed with the competition materials (Omnicampus, `competition.zip`) | Notebook structure and section titles; the 5 baseline features and their preprocessing (label encoding, count encoding, mean imputation, `OWN_CAR_AGE` outlier handling + decade one-hot, `StandardScaler`); the RandomForest baseline model (`max_depth=10`); Section 6.2 code (verbatim) and Section 6.3 code (only the two filename strings changed) | §1–§2.2, §3.1, §4, §5.1, §5.3 (E0), §6.2, §6.3 |
| 2 | GCI Global 2026 In-class Competition README (`README.ipynb`) | GCI organizers | Distributed with the competition materials | Competition rules (no external data, no hand-labeling, reproducibility), metric and submission format | §0, §3.0 |
| 3 | scikit-learn documentation: `StratifiedKFold`, `roc_auc_score`, `RandomForestClassifier` | scikit-learn developers | https://scikit-learn.org/stable/ | API usage for the CV harness | §5.2 |
| 4 | LightGBM documentation: Python API, `early_stopping` callback, parameters (`deterministic`, `force_row_wise`) | Microsoft / LightGBM developers | https://lightgbm.readthedocs.io/ | API usage for `run_cv_lgb` | §5.2 |
| 5 | Home Credit Default Risk (original Kaggle competition, 2018) and its public discussion | Kaggle / Home Credit; community participants | https://www.kaggle.com/c/home-credit-default-risk (link given in the GCI `README.ipynb`) | General, widely shared feature *ideas* only (no code copied, no data used): credit/annuity, annuity/income, credit/income and credit/goods ratios; treating `DAYS_EMPLOYED = 365243` as a placeholder; aggregates of `EXT_SOURCE_*`. Implemented independently in `make_features` | §4.2 (`make_features`), E3–E9, E12 |
| 6 | LightGBM documentation: `LGBMClassifier.fit` (`eval_X` / `eval_y`), `Booster.feature_importance(importance_type="gain")` | Microsoft / LightGBM developers | https://lightgbm.readthedocs.io/ | API usage for `run_cv_lgb` (Sprint 2 API update) and gain importance | §5.2, §5.3, §7.1 |
| 7 | Wilson score interval for a binomial proportion (standard statistics formula) | E. B. Wilson (1927) | — (textbook formula) | 95% confidence intervals for default rates in the EDA | §3.2 (A4, A5, A8) |

## Appendix A. Experiment log
Rendered from `experiments/experiment_log.csv`.

In [ ]:
pd.read_csv("experiments/experiment_log.csv")